# A4-M3 실습 — OpenSearch로 RAG 만들기

- 이 노트북은 `docs/` 폴더, `.env` 파일과 같은 위치에 두고 엽니다.
- `.env.example`을 복사해 `.env`로 이름을 바꾸고 `OPENSEARCH_PASSWORD`(Dashboards 로그인 비밀번호)를 채웁니다. 18탭 LLM까지 하려면 `OPENAI_API_KEY`도.
- 우리 OpenSearch는 **로그인 방식**(https, admin 계정)입니다.
- 위에서부터 **차례로** 실행합니다. 앞 셀에서 만든 이름(`client`, `search` 등)을 뒤 셀이 씁니다.
- 셀 10은 `OPENAI_API_KEY`가 있을 때만 실행합니다. 없으면 셀 11에서 `use_llm=False`로 확인합니다.
- 교안: A4-M3 ‘실습 안내’ 탭과 14~18탭

## 셀 1 · 연결하고 문서를 쪽 단위로 읽기  
교안 15탭

In [ ]:
# pip install opensearch-py sentence-transformers python-dotenv
import os, re
from pathlib import Path
from dotenv import load_dotenv
from opensearchpy import OpenSearch, helpers

load_dotenv()                                         # .env의 OPENSEARCH_PASSWORD (+ OPENAI_API_KEY)
client = OpenSearch(
    hosts=[{"host": "localhost", "port": 9200}],
    http_auth=("admin", os.getenv("OPENSEARCH_PASSWORD")),   # 로그인 방식 (A4-M1 13탭)
    use_ssl=True, verify_certs=False, ssl_show_warn=False,
)
print(client.ping())                                  # True면 연결

DATA_DIR = Path("docs")

def load_doc(path):
    raw = path.read_text(encoding="utf-8")
    head, body = raw.split("\n\n", 1)                 # 첫 빈 줄 위 = 메타데이터, 아래 = 본문
    meta = dict(line.split(": ", 1) for line in head.splitlines())
    meta["year"] = int(meta["year"])
    meta["doc_id"] = path.stem
    pages = re.split(r"=== page (\d+) ===", body)[1:]  # ['11', '본문', '12', '본문', ...]
    return [{**meta, "page": int(pages[i]), "text": pages[i + 1].strip()}
            for i in range(0, len(pages), 2)]

PAGES = [p for f in sorted(DATA_DIR.glob("*.txt")) for p in load_doc(f)]
print("문서", len(list(DATA_DIR.glob("*.txt"))), "개 / 쪽", len(PAGES), "개")
print(PAGES[1]["source"], "p.", PAGES[1]["page"], "|", PAGES[1]["text"][:60], "...")

## 셀 2 · 쪽을 Chunk로 자르기  
교안 15탭

In [ ]:
import math

def chunk_text(text, size=200, overlap=40):
    if overlap >= size:
        raise ValueError("overlap은 size보다 작아야 합니다")
    stride = size - overlap
    n = 1 if len(text) <= size else math.ceil((len(text) - overlap) / stride)
    return [text[i * stride : i * stride + size] for i in range(n)]

chunks = []
for p in PAGES:
    for i, piece in enumerate(chunk_text(p["text"])):
        meta = {k: v for k, v in p.items() if k != "text"}
        chunks.append({"chunk_id": f'{p["doc_id"]}-p{p["page"]}-{i}', "chunk_no": i, "text": piece, **meta})

print("Chunk 수:", len(chunks))
for c in chunks[:3]:
    print(c["chunk_id"], "|", c["text"][:50], "...")

## 셀 3 · 임베딩  
교안 15탭

In [ ]:
from sentence_transformers import SentenceTransformer

MODEL_NAME = "intfloat/multilingual-e5-small"
model = SentenceTransformer(MODEL_NAME)
DIM = model.get_sentence_embedding_dimension()
print("차원:", DIM)                                   # 384

def embed_passages(texts):
    return model.encode(["passage: " + t for t in texts], normalize_embeddings=True).tolist()

def embed_query(text):
    return model.encode("query: " + text, normalize_embeddings=True).tolist()

vectors = embed_passages([c["text"] for c in chunks])
print(len(vectors), "개 ×", len(vectors[0]), "차원")
print("첫 Chunk의 벡터 앞 5개:", [round(x, 3) for x in vectors[0][:5]])

## 셀 4 · 인덱스 만들고 넣기  
교안 15탭

In [ ]:
INDEX = "rag_chunks"
body = {
    "settings": {"index": {"knn": True, "number_of_shards": 1, "number_of_replicas": 0}},
    "mappings": {
        "_meta": {"embedding_model": MODEL_NAME, "dim": DIM},     # 어떤 모델로 만든 벡터인지 기록
        "properties": {
            "chunk_id": {"type": "keyword"}, "doc_id": {"type": "keyword"},
            "source": {"type": "keyword"},   "type": {"type": "keyword"},
            "factory": {"type": "keyword"},  "page": {"type": "integer"},
            "year": {"type": "integer"},     "chunk_no": {"type": "integer"},
            "text": {"type": "text"},
            "embedding": {
                "type": "knn_vector", "dimension": DIM,
                "method": {"name": "hnsw", "engine": "lucene", "space_type": "cosinesimil",
                           "parameters": {"m": 16, "ef_construction": 128}},
            },
        },
    },
}
if client.indices.exists(index=INDEX):
    client.indices.delete(index=INDEX)
client.indices.create(index=INDEX, body=body)

actions = [{"_index": INDEX, "_id": c["chunk_id"], "_source": {**c, "embedding": v}}
           for c, v in zip(chunks, vectors)]
ok, errors = helpers.bulk(client, actions, refresh=True)
print("성공:", ok, "| 실패:", errors, "| count:", client.count(index=INDEX)["count"])

## 셀 5 · 검색 함수  
교안 16탭

In [ ]:
def search(question, k=3, filters=None, ef_search=None):
    knn = {"vector": embed_query(question), "k": k}
    if filters:
        knn["filter"] = {"bool": {"filter": [{"term": {f: v}} for f, v in filters.items()]}}
    if ef_search:
        knn["method_parameters"] = {"ef_search": ef_search}
    res = client.search(index=INDEX, body={
        "size": k,
        "_source": {"excludes": ["embedding"]},
        "query": {"knn": {"embedding": knn}},
    })
    return [{"score": round(h["_score"], 4), **h["_source"]} for h in res["hits"]["hits"]]

def show(hits):
    for h in hits:
        print(f'  {h["score"]}  [{h["source"]} p.{h["page"]}]  {h["text"][:40]}...')

for q in ["기계에 손이 끼지 않게 하려면?", "불나면 어디로 나가요?", "포크리프트 속도 제한이 있나요?"]:
    print(q); show(search(q))

## 셀 6 · 필터  
교안 16탭

In [ ]:
q = "작업 중 사고를 예방하려면?"
print("필터 없음");               show(search(q))
print("화성 공장 · 2026년만");      show(search(q, filters={"factory": "화성", "year": 2026}))
print("법령만");                  show(search(q, filters={"type": "법령"}))

## 셀 7 · ef_search  
교안 16탭

In [ ]:
show(search("방호장치", k=5))
show(search("방호장치", k=5, ef_search=200))   # 10탭: 검색할 때 탐색 폭

## 셀 8 · Hybrid 검색  
교안 17탭

In [ ]:
def make_pipeline(name, w_keyword, w_vector):
    body = {"phase_results_processors": [{"normalization-processor": {
        "normalization": {"technique": "min_max"},
        "combination": {"technique": "arithmetic_mean",
                        "parameters": {"weights": [w_keyword, w_vector]}},
    }}]}
    client.transport.perform_request("PUT", f"/_search/pipeline/{name}", body=body)

make_pipeline("hybrid-pipe", 0.3, 0.7)

def hybrid_search(question, k=3, pipeline="hybrid-pipe"):
    body = {
        "size": k,
        "_source": {"excludes": ["embedding"]},
        "query": {"hybrid": {"queries": [
            {"match": {"text": question}},                                   # 키워드 (weights 첫 번째)
            {"knn": {"embedding": {"vector": embed_query(question), "k": k}}},  # 벡터 (weights 두 번째)
        ]}},
    }
    res = client.search(index=INDEX, body=body, params={"search_pipeline": pipeline})
    return [{"score": round(h["_score"], 4), **h["_source"]} for h in res["hits"]["hits"]]

q = "제38조"
print("벡터만");  show(search(q))
print("Hybrid");  show(hybrid_search(q))

## 셀 9 · 근거와 프롬프트 (키 없이 동작)  
교안 18탭

In [ ]:
def build_context(hits):
    return "\n".join(f'[{i}] ({h["source"]} p.{h["page"]}) {h["text"]}'
                     for i, h in enumerate(hits, 1))

def build_prompt(question, hits):
    return (
        "다음 [근거]만 사용해 질문에 한국어로 답하세요.\n"
        "근거에 없는 내용은 '근거 문서에서 찾을 수 없습니다'라고 답하세요.\n"
        "답변 끝에 사용한 근거 번호를 [1]처럼 표시하세요.\n\n"
        f"[근거]\n{build_context(hits)}\n\n[질문]\n{question}"
    )

question = "프레스 작업 중 손 끼임을 어떻게 예방하나요?"
hits = hybrid_search(question, k=3)          # STEP 2~3: Embedding + OpenSearch 검색
print(build_prompt(question, hits))         # STEP 4: LLM에 넘어갈 내용을 먼저 눈으로 확인

## 셀 10 · LLM 호출 (API 키 있을 때만)  
교안 18탭

In [ ]:
# pip install openai   ·   환경변수 OPENAI_API_KEY 필요
from openai import OpenAI

LLM_MODEL = "gpt-4o-mini"    # 강의 시점에 사용 가능한 모델명으로 바꿔 쓰세요
llm = OpenAI()

def generate(prompt):
    resp = llm.chat.completions.create(
        model=LLM_MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
    )
    return resp.choices[0].message.content

print(generate(build_prompt(question, hits)))

## 셀 11 · rag_answer 완성  
교안 18탭

In [ ]:
MIN_SCORE = 0.90     # 벡터 점수 기준 — 출발값일 뿐, 미션 6에서 직접 정합니다

def rag_answer(question, k=3, filters=None, use_llm=True):
    vec_hits = search(question, k=k, filters=filters)
    if not vec_hits or vec_hits[0]["score"] < MIN_SCORE:
        return "관련 근거를 찾지 못했습니다. 담당자에게 문의해 주세요.", []
    prompt = build_prompt(question, vec_hits)
    answer = generate(prompt) if use_llm else prompt
    return answer, [(h["source"], h["page"]) for h in vec_hits]

for q in ["불나면 엘리베이터 타도 되나요?", "오늘 점심 메뉴 추천해 줘"]:
    answer, sources = rag_answer(q, use_llm=False)    # 키가 있으면 True
    print("Q:", q); print(answer[:200]); print("출처:", sources); print("---")

## 미션 (교안 19탭)
아래에 셀을 추가해 미션을 진행합니다.

In [ ]:
# 미션 1부터 여기에 이어서 작성
